# 02 — Join MovieLens + IMDb (Step 2)

**What:** Joins MovieLens 25M (base) to IMDb's `title.basics` via a reconstructed `tconst`, applies the decided edge-case rules, and saves one cleaned catalog table.

**Why:** Everything from Step 3 (EDA) onward needs one row per movie with a reliable year, genre, and type — this notebook is where that gets settled, with every call logged with its evidence rather than assumed.

**Decisions applied (see docs/DECISIONS.md for the full log with counts):**
- Unmatched IMDb IDs (129 / 62,423, 0.2%): kept, with null IMDb columns.
- `titleType`: strict filter to `movie` only, but **only for matched rows** — the 129 unmatched rows have no `titleType` to check and are kept regardless. Drops 9,449 matched-but-non-movie rows (15.1% of matched).
- Year: trust IMDb `startYear` as canonical; fall back to the year parsed from the MovieLens title for the 129 unmatched (and any other row missing `startYear`).
- Genres: normalize MovieLens's `"(no genres listed)"` sentinel to null, then backfill from IMDb's `genres` where available (recovers 3,673 of 3,699 null rows in the kept set).

**Inputs:**
- `data/raw/movielens/ml-25m/{movies,links,ratings}.csv`
- `data/raw/imdb/<snapshot_date>/title.basics.tsv.gz`

**Output:** `data/processed/movies_joined.parquet` + `metadata.json`

Safe to re-run: the Save stage checks for the output file first and skips the join if it already exists and inputs haven't changed.

## Config
All paths. Uses the same `data/raw/` snapshot that notebook 01 landed.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ML_DIR = Path("..") / "data" / "raw" / "movielens" / "ml-25m"
IMDB_SNAPSHOT_DATE = "2026-10-05"  # matches the snapshot notebook 01 landed
IMDB_DIR = Path("..") / "data" / "raw" / "imdb" / IMDB_SNAPSHOT_DATE

PROCESSED_DIR = Path("..") / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "movies_joined.parquet"
METADATA_PATH = PROCESSED_DIR / "movies_joined.metadata.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Stage: Load
Loads MovieLens `movies.csv`, `links.csv`, `ratings.csv` (movieId only, to flag zero-rating movies) and IMDb `title.basics`. Column names that collide across sources (`genres`) are renamed on load so the merge doesn't silently overwrite one with the other.

In [2]:
movies = pd.read_csv(ML_DIR / "movies.csv").rename(columns={"genres": "ml_genres"})
links = pd.read_csv(ML_DIR / "links.csv")
rated_movie_ids = set(pd.read_csv(ML_DIR / "ratings.csv", usecols=["movieId"])["movieId"].unique())

basics = pd.read_csv(
    IMDB_DIR / "title.basics.tsv.gz", sep="\t", compression="gzip", na_values="\\N", low_memory=False
).rename(columns={"genres": "imdb_genres"})

print(f"movies: {len(movies):,} rows")
print(f"links: {len(links):,} rows")
print(f"basics: {len(basics):,} rows")

movies: 62,423 rows
links: 62,423 rows
basics: 12,838,880 rows


## Stage: Build join key
`links.csv` stores `imdbId` as a bare integer; IMDb's `tconst` is `"tt" + zero-padded digits`. `zfill(7)` reconstructs it correctly for every row, including the 131 movies whose ID already exceeds 7 digits (zfill is a no-op there, not a truncation).

In [3]:
links["tconst"] = "tt" + links["imdbId"].astype(str).str.zfill(7)
print(links[["movieId", "imdbId", "tconst"]].head(3))

   movieId  imdbId     tconst
0        1  114709  tt0114709
1        2  113497  tt0113497
2        3  113228  tt0113228


## Stage: Join
Left-joins MovieLens -> links -> IMDb basics, so every MovieLens movie is kept through this stage regardless of match status (filtering happens next, deliberately, as its own stage).

In [4]:
df = movies.merge(links, on="movieId", how="left").merge(basics, on="tconst", how="left")
assert len(df) == len(movies), "join should not change row count before filtering"
print(f"joined: {len(df):,} rows")

joined: 62,423 rows


## Stage: Apply titleType filter
Drops rows that matched an IMDb title but aren't `titleType == "movie"` (e.g. `tvEpisode`, `short`, `tvSeries`). Unmatched rows (no IMDb row at all, so `titleType` is null) are kept regardless — they're missing data, not failing a type check.

In [5]:
is_matched = df["titleType"].notna()
is_movie_type = df["titleType"] == "movie"
keep_mask = (~is_matched) | is_movie_type

n_dropped = (~keep_mask).sum()
print(f"dropping {n_dropped:,} rows (matched, non-movie titleType)")
print(df.loc[~keep_mask, "titleType"].value_counts())

df = df[keep_mask].copy()
print(f"\nremaining: {len(df):,} rows")

dropping 9,449 rows (matched, non-movie titleType)
titleType
tvMovie         3601
short           2791
video           1535
tvSpecial        527
tvMiniSeries     454
tvEpisode        368
tvShort           88
tvSeries          85
Name: count, dtype: int64

remaining: 52,974 rows


## Stage: Resolve year
Trusts IMDb `startYear` as the canonical year. Falls back to the year parsed from the MovieLens title (`"Toy Story (1995)"` -> `1995`) only where `startYear` is missing — covers the 129 unmatched rows plus any other row IMDb didn't record a year for.

In [6]:
ml_year = df["title"].str.extract(r"\((\d{4})\)\s*$")[0].astype("float")
df["ml_year"] = ml_year
df["year"] = df["startYear"].fillna(df["ml_year"])

print(f"rows with no year at all (both sources missing): {df['year'].isna().sum():,}")
print(f"rows filled from MovieLens fallback: {(df['startYear'].isna() & df['year'].notna()).sum():,}")

rows with no year at all (both sources missing): 3
rows filled from MovieLens fallback: 132


## Stage: Resolve genres
Converts MovieLens's `"(no genres listed)"` sentinel string to a real null (consistent with IMDb's `\N` -> NaN), then backfills from IMDb's `genres` wherever MovieLens has none.

In [7]:
df["ml_genres"] = df["ml_genres"].replace("(no genres listed)", pd.NA)
n_null_before = df["ml_genres"].isna().sum()

df["genres"] = df["ml_genres"].fillna(df["imdb_genres"])

n_null_after = df["genres"].isna().sum()
print(f"null genres before backfill: {n_null_before:,}")
print(f"null genres after IMDb backfill: {n_null_after:,}")
print(f"recovered: {n_null_before - n_null_after:,}")

null genres before backfill: 3,699
null genres after IMDb backfill: 26
recovered: 3,673


## Stage: Add derived flags
`has_ratings` (for the zero-rating movies, kept per the quality-tier AND-logic in PROJECT.md §4.1 — age alone never excludes a movie) and `is_imdb_matched` (for anything still missing IMDb columns, flagged for TMDB backfill at Step 4+).

In [8]:
df["has_ratings"] = df["movieId"].isin(rated_movie_ids)
df["is_imdb_matched"] = df["titleType"].notna()

print(f"movies with zero ratings (kept): {(~df['has_ratings']).sum():,}")
print(f"movies still unmatched to IMDb (kept, flagged): {(~df['is_imdb_matched']).sum():,}")

movies with zero ratings (kept): 2,928
movies still unmatched to IMDb (kept, flagged): 129


## Stage: Select final columns
Keeps both the resolved fields and the original per-source fields (`ml_year`/`startYear`, `ml_genres`/`imdb_genres`) rather than discarding them — per the year-conflict decision, nothing gets thrown away silently.

In [9]:
final = df[[
    "movieId", "tconst", "tmdbId", "title", "genres", "year", "has_ratings", "is_imdb_matched",
    "titleType", "runtimeMinutes", "ml_genres", "imdb_genres", "ml_year", "startYear",
]].rename(columns={"startYear": "imdb_year"})

print(final.shape)
final.head(3)

(52974, 14)


,movieId,tconst,tmdbId,title,genres,year,has_ratings,is_imdb_matched,titleType,runtimeMinutes,ml_genres,imdb_genres,ml_year,imdb_year
0,1,tt0114709,862.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,True,True,movie,81,Adventure|Animation|Children|Comedy|Fantasy,"Adventure,Animation,Comedy",1995.0,1995.0
1,2,tt0113497,8844.0,Jumanji (1995),Adventure|Children|Fantasy,1995.0,True,True,movie,104,Adventure|Children|Fantasy,"Adventure,Comedy,Family",1995.0,1995.0
2,3,tt0113228,15602.0,Grumpier Old Men (1995),Comedy|Romance,1995.0,True,True,movie,101,Comedy|Romance,"Comedy,Romance",1995.0,1995.0


## Stage: Save
Writes the joined table to Parquet in `data/processed/`, plus a metadata JSON per the save-expensive-work rules. Skips the whole notebook's heavy lifting on re-run if the output already exists — delete it manually first if you want to force a rebuild after an input or decision changes.

In [10]:
if OUTPUT_PATH.exists():
    print(f"{OUTPUT_PATH} already exists, not overwriting. Delete it first to force a rebuild.")
else:
    final.to_parquet(OUTPUT_PATH, index=False)
    metadata = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source_movielens_metadata": str(Path("..") / "data" / "raw" / "movielens" / "metadata.json"),
        "source_imdb_snapshot": IMDB_SNAPSHOT_DATE,
        "row_count": len(final),
        "decisions_applied": [
            "unmatched IMDb IDs kept with null IMDb columns (129 rows)",
            "titleType strict filter to 'movie', applied only to matched rows (dropped 9449 rows)",
            "year: IMDb startYear canonical, MovieLens title-year fallback",
            "genres: '(no genres listed)' normalized to null, backfilled from IMDb",
        ],
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2))
    print(f"Saved {len(final):,} rows to {OUTPUT_PATH}")
    print(f"Metadata written to {METADATA_PATH}")

../data/processed/movies_joined.parquet already exists, not overwriting. Delete it first to force a rebuild.


## Summary

In [11]:
print("Join summary (Step 2)")
print("=" * 40)
print(f"Output: {OUTPUT_PATH} ({'OK' if OUTPUT_PATH.exists() else 'MISSING'})")
print(f"Rows: {len(final):,} (started from {len(movies):,} MovieLens movies, dropped {n_dropped:,} non-movie titleType)")
print(f"Zero-rating movies kept: {(~final['has_ratings']).sum():,}")
print(f"Still unmatched to IMDb (flagged): {(~final['is_imdb_matched']).sum():,}")
print(f"Null genres remaining after backfill: {final['genres'].isna().sum():,}")
print(f"Null year remaining: {final['year'].isna().sum():,}")

Join summary (Step 2)
Output: ../data/processed/movies_joined.parquet (OK)
Rows: 52,974 (started from 62,423 MovieLens movies, dropped 9,449 non-movie titleType)
Zero-rating movies kept: 2,928
Still unmatched to IMDb (flagged): 129
Null genres remaining after backfill: 26
Null year remaining: 3
